# Cryptography (CC4017) -- Week 11


### Q1: Demonstrating forgeability of plain RSA
A key property of digital signatures is that of existential unforgeability, which is expressed as a security
game as follows.

### Existential Unforgeability

1 - We generate a keypair (sk, pk)

2 - We give pk to the adversary, and allow him to request signatures of arbitrary messages. This means
that the adversary is free to request signatures of anything, but every signed message is recorded.

3 - The adversary must then produce a pair (m, s). He wins the game if: i.) s is a valid signature for m
and ii.) the signature of m was not requested in step 2.

### Plain RSA signature scheme
Plain RSA as a signature scheme sets (e, n) as the public key and (d, n) as the private key. Signatures
of m are md and validation of (m, s) is checking if m = se
It is easy to see that plain RSA is not existentially unforgeable. Indeed, anyone can produce forgery
(1, 1), as 1^e = 1 regardless of the value of e. However the issues of signing with plain RSA go above
and beyond this singular case. Let’s go for a slightly more challenging scenario.

Question: Describe how an adversary can produce a valid forgery for the following experiment with
probability 1.
1
### Existential unforgeability with a twist:

1 - We generate a keypair (sk, pk) and select a bad value b.

2 - We give pk and b to the adversary, and allow him to request signatures of arbitrary messages. This
means that the adversary is free to request signatures of anything, but every signed message is
recorded.

3 - The adversary must then produce a signature s. He wins the game if i.) s is a valid signature for b
and ii.) the signature of b was not requested in step 2.

In order to produce a valid signature with cerntainty for b the adversary uses the following strategy:
    
    1. Find two values x and y, such xy ≡ 1 mod n
    
    2. The adversary requests the oracle to sign x, producing sx = x^d mod n

    3. After the adversary computes y*b mod n, and requests its signature as well, producing sy = (y*b)^d mod n  

    4. The valid signature sb is now the result of multiplying sy * sx.

To verify if the signature was forged successfully, we need to verify that b ≡ sb^e mod n

sb^e = (sx*sy)^e mod n , which simplifies to:

(x^d * yb^d)^e mod n, which is equivalent to:

sb^e = ((xyb)^d)^e mod n = xyd^de, d*e ≡ 1 mod Φ(n), and due to (xyb)^de = (yb)^(1 + k *  Φ(n)) = (yb) * (yb)^{k phi(n)}.Using the fact that for any a. a^ Φ(n) ≡ 1 mod n, we obtain:

sb^e = xyb mod n, and from xy ≡ 1 mod n it follows that:

sb^e = b mod n, implying b ≡ sb^e mod n, proving that sb is a valid signature for b

## Q2: Shamir Secret Sharing


In the extra work of class 1, we overviewed a technique for secret sharing that allowed messages to be
split in parts, such that it could only be reconstructed if a subset of participants agreed to participate i.e. revealed their “shares” of the original value.

Shamir Secret Sharing is based on polynomial interpolation over finite fields. Consider a point (0, 1).
It is easy to observe that there is an overwhelming number of polynomials p of degree 1 such that p(0) = 1.

E.g.

p(x) = x + 1; p(x) = 2x + 1; p(x) = 4x + 1; etc.

However, when given two points (0, 1) and (1, 3) there is only one polynomial p for which these points are valid: p(x) = 2x + 1

The intuition for Shamir Secret Sharing is that secrets are represented as polynomials, and shares are represented as points in said polynomial. As such, one can very flexibly select the number of necessary shares and the threshold for secret reconstruction by adjusting the parameters of the system.

We will consider integer polynomial coefficients, and standard integer arithmetic. A practical implementation of shamir secret sharing requires computation over a finite field to ensure privacy. However, for this didatic exercise, we can disregard this very simple adaptation, as the intuitions we want to understand can be observed in integer arithmetic, which is slightly more straightfoward to explore.

### P1: 
Implement the secret sharing function that takes value x and produces a set of four shares x1, x2, x3, x4 such that any three can be used to reconstruct the original value.
This will entail responding to the following problems:

1- What degree should the polynomial be, so that three points are sufficient to reconstruct the value,
but two points can never be enough?

2- How can we generate a polynomial f of that degree, such that f (0) = x?

Before writting any code the two questions should be answered. Firstly, if we consider the polynomial to be the secret, and assum that, as any polynomial, it discribes a function, in N in this case, we know, as mathematical fact, that the number of points necessary to describe a function is the degree + 1. Since we want to generate a polynomial that can be reconstructed by 3 points, we need a polynomial of degree 2. Polynomials of degree 2 are of the form ax²+bx+c, therefore, if we want f(0) to be equal to the secret x, we simply need to say that ,using y to make it easier to understand, f(y) =ay² + by + x , with x being the secret, that will yield. f(0) = a * 0² + b * 0 + x = x.

in this case, we generate a random a and b value, and plug the secret as the c value, making it so the secret can be obtained by computing f(0), but also making it so we need to first reconstruct f through 3 points.

In [17]:
import random
from sympy import symbols

def generate_polynomial(secret, degree=2):
    x = symbols('x')
    coefficients = [random.randint(1, 128) for _ in range(degree)] 
    coefficients.append(secret)  
    polynomial = sum(c * (x ** i) for i, c in enumerate(reversed(coefficients)))  
    return polynomial

def shamir_secret_sharing(secret, num_shares=4, threshold=3):
    degree = threshold - 1  
    polynomial = generate_polynomial(secret, degree)
    x = symbols('x')
    shares = [(i, polynomial.subs(x, i)) for i in range(1, num_shares + 1)] 
    return shares, polynomial


secret = 378
shares, polynomial = shamir_secret_sharing(secret)
print("Shares:", shares)
print("Polynomial:", polynomial)
    

Shares: [(1, 509), (2, 760), (3, 1131), (4, 1622)]
Polynomial: 60*x**2 + 71*x + 378


### P2: 
Implement a function for polynomial interpolation, that takes n points, and recovers the only polynomial of degree n − 1 that contains those points.

Show how this allows for the secret to be recovered:

1- Generate a polynomial f for secret number 1001 and shares (points) for that secret.

2- Take only the minimum amount of necessary shares (randomly selected) and retrieve the polynomial
f ′

3- Show that, for x = 0, f ′(x) = 1001, and thus the secret can be recovered


Polynomial Interpolation
To reconstruct the polynomial from points, we use Lagrange Interpolation:

- This determines the unique polynomial passing through all the given points.

- Use only the minimum number of shares (n=3).
- Interpolate the polynomial and evaluate it at x=0 to recover the secret.

In [18]:
def interpolate_polynomial(points):
    x = symbols('x')
    terms = []
    for i, (xi, yi) in enumerate(points):
        li = 1  
        for j, (xj, _) in enumerate(points):
            if i != j:
                li *= (x - xj) / (xi - xj)  #
        terms.append(yi * li)
    polynomial = sum(terms) 
    return polynomial.expand()

def recover_secret(shares):
    polynomial = interpolate_polynomial(shares)
    secret = polynomial.subs(symbols('x'), 0)  
    return secret

secret = 1001
shares, polynomial = shamir_secret_sharing(secret)
subset_of_shares = random.sample(shares, 3)  
recovered_polynomial = interpolate_polynomial(subset_of_shares)
recovered_secret = recover_secret(subset_of_shares)

print("Recovered Polynomial:", recovered_polynomial)
print("Recovered Secret:", recovered_secret)

Recovered Polynomial: 59*x**2 + 126*x + 1001
Recovered Secret: 1001


### P3:

Use what you implemented to test the following:
1- Generate a polynomial f for secret number 100 and shares (points) for that secret: x1, x2, x3, x4.

2- Generate another polynomial g for secret number 550 and shares (points) for that secret y1, y2, y3, y4

3- Calculate z1 = x1 + y1; z2 = x2 + y2 and z3 = x3 + y3

4- Use your secret recovery method using points z1, z2, z3. What can you conclude from the result?

Justify what happened.

In [19]:

def add_shares(shares_f, shares_g):
    combined_shares = [(x, y1 + y2) for (x, y1), (_, y2) in zip(shares_f, shares_g)]
    return combined_shares

shares_f, _ = shamir_secret_sharing(100)  
shares_g, _ = shamir_secret_sharing(550)  
# Combine shares
combined_shares = add_shares(shares_f, shares_g)

recovered_polynomial_combined = interpolate_polynomial(combined_shares[:3])
combined_secret = recover_secret(combined_shares[:3])

print("Combined Polynomial:", recovered_polynomial_combined)
print("Combined Secret:", combined_secret)

Combined Polynomial: 95*x**2 + 156*x + 650
Combined Secret: 650


When generating shares, the points on the list belong to a curve defined by the equation f(x)=ax²+bx+c , where c is the secret value. When adding the points from x and y, we obtain a new set of points that, this time belong to the curve defined by f(x) + f(y) = ax²+bx+100 + jx²+kx+550 .

These new points belong to this newly formed curve, given by the formula f(z) = ax² + jx² + bx + kx + 650. We can conclude that the secret in this curve is now 650, that can easily be proven by calculating f(0) where we obtain the secret that's 650 and by the code demonstration.